In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from statsbombpy import sb
from src.displacement import add_displacement, compute_displacement

In [2]:
# Quick sanity check: compute displacement for a single known pair
result = compute_displacement([110.6, 0.1], [108.0, 0.1])
print(f"compute_displacement: {result}")

compute_displacement: {'creep_m': 2.275, 'creep_x_m': 2.275, 'creep_y_m': 0.0}


In [3]:
# Load one Euro 2024 match
matches = sb.matches(competition_id=55, season_id=282)
match_id = matches.match_id.iloc[0]

home = matches.loc[matches.match_id == match_id, 'home_team'].iloc[0]
away = matches.loc[matches.match_id == match_id, 'away_team'].iloc[0]
print(f"Match: {home} vs {away}  (id={match_id})")

events = sb.events(match_id=match_id)
print(f"Events loaded: {len(events)}")

Match: Netherlands vs England  (id=3942819)
Events loaded: 3485


/Users/mingeonsung/sports-analytics/soccer/throw-in-analysis/venv/lib/python3.10/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/Users/mingeonsung/sports-analytics/soccer/throw-in-analysis/venv/lib/python3.10/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [4]:
# Compute displacement for every throw-in in the match
paired = add_displacement(events)
valid  = paired.dropna(subset=['creep_m'])

print(f"Throw-ins: {len(paired)}  |  paired with exit location: {len(valid)} ({len(valid)/len(paired):.0%})")

Throw-ins: 25  |  paired with exit location: 7 (28%)


In [5]:
# View displacement results
valid[['team', 'minute', 'throw_location', 'exit_location', 'creep_m', 'creep_x_m', 'creep_y_m']]

,team,minute,throw_location,exit_location,creep_m,creep_x_m,creep_y_m
7,Netherlands,26,"[85.1, 80.0]","[103.6, 80.0]",16.188,-16.188,0.000
11,England,46,"[35.7, 80.0]","[44.0, 80.0]",7.262,-7.262,0.000
12,England,48,"[80.0, 80.0]","[73.0, 80.0]",6.125,6.125,0.000
16,Netherlands,54,"[36.2, 80.0]","[50.2, 80.0]",12.250,-12.250,0.000
19,England,58,"[57.0, 0.1]","[55.2, 0.0]",1.577,1.575,0.085
20,Netherlands,59,"[30.1, 0.1]","[29.400000000000006, 0.0]",0.618,0.612,0.085
22,Netherlands,74,"[37.1, 80.0]","[29.5, 80.0]",6.650,6.650,0.000


In [6]:
# Summary stats per team
valid.groupby('team')[['creep_m', 'creep_x_m', 'creep_y_m']].agg(['mean', 'count']).round(2)

creep_m       creep_x_m       creep_y_m      
               mean count      mean count      mean count
team                                                     
England        4.99     3      0.15     3      0.03     3
Netherlands    8.93     4     -5.29     4      0.02     4